<a href="https://colab.research.google.com/github/Michael-AI-Dam/Flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Michael-AI-Dam/Flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

**Findings:** All four fields are heavy-tailed, confirmed by mean/median ratios
far from 1.0. `impressions_last_30d` has a mean/median ratio of 9.36 (mean 1,489
vs. median 159) — a small number of pages (max 238,796) pull the average far
above what a typical page sees. `clicks_last_30d` is even more extreme: the
median is 0 (75% of pages get 2 clicks or fewer in 30 days), so the mean/median
ratio is undefined (division by zero) — this alone tells us most pages get
essentially no clicks, and the distribution is dominated by a small number of
high-performing outliers.

`ctr` also confirms the skill file's "already a percentage" gotcha directly:
median CTR is 0.08%, mean is 0.52%, and the max is 100% — real click-through
rates this low (median under 0.1%) only make sense if `ctr` is a percentage
value already, not a 0–1 fraction (0.08 meaning 0.08%, not 8%). This matches
what the data skill flagged and confirms our Week 1 correction was right.

`avg_position` is comparatively mild (ratio 1.49, median 11.4) — still
right-skewed but nowhere near as extreme as impressions/clicks.

**Conclusion:** impressions, clicks, and CTR all require log1p or rank-based
(Spearman) handling before any correlation — confirmed directly by these ratios,
not just assumed from the skill file's general warning about web traffic data.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("https://raw.githubusercontent.com/Michael-AI-Dam/Flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv")

# avg_position == 0 means "no data" — exclude for any position-based work
df_valid = df[df["avg_position"] > 0].copy()

for col in ["impressions_last_30d", "clicks_last_30d", "ctr", "avg_position"]:
    print(f"--- {col} ---")
    print(df_valid[col].describe())
    print(f"mean/median ratio: {df_valid[col].mean() / df_valid[col].median():.2f}")
    print()

--- impressions_last_30d ---
count     28795.000000
mean       1488.787567
std        5753.022143
min           0.000000
25%          16.000000
50%         159.000000
75%         831.000000
max      238796.000000
Name: impressions_last_30d, dtype: float64
mean/median ratio: 9.36

--- clicks_last_30d ---
count    28795.000000
mean         5.140267
std         24.403248
min          0.000000
25%          0.000000
50%          0.000000
75%          2.000000
max       1176.000000
Name: clicks_last_30d, dtype: float64
mean/median ratio: inf

--- ctr ---
count    28795.000000
mean         0.519662
std          3.232606
min          0.000000
25%          0.000000
50%          0.080000
75%          0.300000
max        100.000000
Name: ctr, dtype: float64
mean/median ratio: 6.50

--- avg_position ---
count    28795.000000
mean        17.026268
std         15.152439
min          0.100000
25%          6.700000
50%         11.400000
75%         22.900000
max        245.000000
Name: avg_position, d

/tmp/ipykernel_549/4217173934.py:12: RuntimeWarning: divide by zero encountered in scalar divide
  print(f"mean/median ratio: {df_valid[col].mean() / df_valid[col].median():.2f}")


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

**Test 1 — claim:** "Pages that rank higher (lower avg_position) get more impressions."
**Test:** grouped median impressions by position_tier, with n shown.

| position_tier | median_impressions | n |
|---|---|---|
| deep | 50.0 | 1,319 |
| page_1 | 198.0 | 11,814 |
| page_3_5 | 178.0 | 7,242 |
| striking | 175.0 | 7,304 |
| top_3 | 3.0 | 1,116 |

**Verdict: OPPOSITE.** The pattern is not monotonic and the extreme case directly
contradicts the claim: `top_3` (the best position tier) has the *lowest* median
impressions (3) in the entire table — even lower than `deep` (the worst tier,
median 50). This is almost certainly a tier-labeling or definitional issue worth
flagging rather than a real finding — "top_3" may represent a rare or narrow
result type (e.g. featured snippets, which get few raw impressions by design)
rather than "best-ranked regular results." In practice: don't assume tier
ordering implies a smooth position → impressions relationship without checking
what each tier actually represents.

**Test 2 — claim:** "Fresher content (lower days_since_last_update) gets a better CTR."
**Test:** weighted CTR (total clicks / total impressions) by freshness_tier, with n shown.

| freshness_tier | weighted_ctr | n |
|---|---|---|
| 0-30 | 0.003574 | 19,300 |
| 31-90 | 0.001984 | 175 |
| 91-180 | 0.003322 | 9,162 |
| 181+ | 0.005316 | 158 |

**Verdict: MIXED.** Comparing the two largest, most trustworthy buckets
(0-30 vs. 91-180, both with thousands of rows), the freshest content does have
a marginally higher CTR (0.36% vs. 0.33%) — weak support for the claim. But the
two smallest buckets (31-90 and 181+, both just above the sample-size floor at
n=175 and n=158) show the opposite pattern, with 181+ (the stalest content)
posting the *highest* CTR of all four tiers. Given how thin those two buckets
are relative to the other two, this isn't strong enough evidence either way.
In practice: freshness alone is not a reliable standalone CTR predictor — the
small-bucket reversal means this needs a larger sample before a content team
should act on "refresh = better CTR" as a rule.

**Test 3 — claim:** "Longer pages (higher word_count) get more impressions."
**Test:** grouped median impressions by word_count_tier, with n shown.

| word_count_tier | median_impressions | n |
|---|---|---|
| <1000 | 2.0 | 719 |
| 1000-2000 | 21.0 | 3,392 |
| 2000-3500 | 192.0 | 11,141 |
| 3500+ | 299.0 | 5,857 |

**Verdict: CONFIRMED.** The relationship is clean and monotonic — median
impressions rise at every step as word count increases, from 2 (shortest pages)
to 299 (longest). All buckets are comfortably above the sample-size floor. In
practice: this supports prioritizing longer-form content when impressions are
the goal, though this test alone doesn't establish word count *causes* more
impressions (longer pages may also cover broader topics, which independently
drives more search visibility).

In [2]:
# Test 1: position vs. impressions (log-safe via median, not mean)
test1 = df_valid.groupby("position_tier").agg(
    median_impressions=("impressions_last_30d", "median"),
    n=("impressions_last_30d", "size")
).reset_index()
print(test1)
print("\nFloor check: all buckets >= 50 rows?" , (test1["n"] >= 50).all())

  position_tier  median_impressions      n
0          deep                50.0   1319
1        page_1               198.0  11814
2      page_3_5               178.0   7242
3      striking               175.0   7304
4         top_3                 3.0   1116

Floor check: all buckets >= 50 rows? True


In [3]:
# Test 2: freshness vs. CTR — weighted CTR, not mean-of-rates (per skill's trap)
test2 = df_valid.groupby("freshness_tier").apply(
    lambda g: pd.Series({
        "weighted_ctr": g["clicks_last_30d"].sum() / g["impressions_last_30d"].sum(),
        "n": len(g)
    })
).reset_index()
print(test2)
print("\nFloor check: all buckets >= 50 rows?", (test2["n"] >= 50).all())

  freshness_tier  weighted_ctr        n
0           0-30      0.003574  19300.0
1           181+      0.005316    158.0
2          31-90      0.001984    175.0
3         91-180      0.003322   9162.0

Floor check: all buckets >= 50 rows? True


/tmp/ipykernel_549/4097379500.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test2 = df_valid.groupby("freshness_tier").apply(


In [4]:
# Test 3: word count vs. impressions
test3 = df_valid.groupby("word_count_tier").agg(
    median_impressions=("impressions_last_30d", "median"),
    n=("impressions_last_30d", "size")
).reset_index()
print(test3)
print("\nFloor check: all buckets >= 50 rows?", (test3["n"] >= 50).all())

  word_count_tier  median_impressions      n
0       1000-2000                21.0   3392
1       2000-3500               192.0  11141
2           3500+               299.0   5857
3           <1000                 2.0    719

Floor check: all buckets >= 50 rows? True


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**Claim (behind the CTR-fix flag):** "Pages with CTR well below their position-tier
peers are a real, identifiable group — not just noise."
**Test:** compare the share of pages below their position-tier median CTR, with n
shown, using rank-based comparison (not raw correlation) since CTR/impressions are
heavy-tailed.
**Verdict: MIXED**

The middle three tiers (page_1, page_3_5, striking) behave exactly as expected —
roughly 49–50% of pages fall below their tier's median CTR, which is what a
clean 50/50 split should look like. But the two edge tiers (deep, top_3) show
0% below median, which isn't a real effect — it's a median-tie artifact: with
smaller bucket sizes (1,116–1,319 rows) and CTR values that repeat often, many
rows land exactly ON the median rather than below it, so the "below median"
flag never triggers for them.

The Spearman correlation (-0.234, p<0.0001) gives a cleaner overall read: it's
a real but weak negative relationship — worse position (higher avg_position)
is associated with lower CTR, which is what the flag's underlying assumption
predicts. But -0.234 is a modest effect, not a strong one, and the tier-level
test shows the "below vs. above median" framing breaks down at the extremes
(deep and top_3), where tie-heavy distributions make a binary split
uninformative rather than confirming or refuting the pattern.

**In practice:** the CTR-fix flag's core assumption holds directionally
(position and CTR are related, in the expected direction), but the relationship
is weak enough that CTR-vs-position alone shouldn't be treated as a high-confidence
trigger — it's one useful signal among several, not a strong standalone predictor.
The edge-tier tie artifact is also worth flagging to a mentor: it may mean `ctr`
values are coarser/less granular than expected within those tiers.

In [7]:
# Flag-linked test: CTR vs. position-tier peers (behind the CTR-fix logic flag)
df_valid["ctr_tier_median"] = df_valid.groupby("position_tier")["ctr"].transform("median")
df_valid["below_tier_median"] = df_valid["ctr"] < df_valid["ctr_tier_median"]

flag_test = df_valid.groupby("position_tier").agg(
    n=("ctr", "size"),
    pct_below_median=("below_tier_median", "mean")
).reset_index()
print(flag_test)
print("\nFloor check: all buckets >= 50 rows?", (flag_test["n"] >= 50).all())

# Spearman correlation as a sanity check (rank-based, heavy-tail safe)
from scipy.stats import spearmanr
corr, pval = spearmanr(df_valid["avg_position"], df_valid["ctr"])
print(f"\nSpearman correlation (position vs ctr): {corr:.3f}, p={pval:.4f}")

  position_tier      n  pct_below_median
0          deep   1319          0.000000
1        page_1  11814          0.496868
2      page_3_5   7242          0.491853
3      striking   7304          0.499315
4         top_3   1116          0.000000

Floor check: all buckets >= 50 rows? True

Spearman correlation (position vs ctr): -0.234, p=0.0000


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

Word count is the one clearly confirmed signal here — longer content reliably
gets more impressions, so it's safe to prioritize when writing new pages.
Position and freshness are both weaker or messier than the popular story
suggests: the position-tier test came back OPPOSITE due to what looks like a
tier-labeling issue (worth a follow-up before trusting tier-based rules), and
freshness only weakly correlates with CTR once bucket size is accounted for.
The CTR-fix flag itself is directionally supported (Spearman confirms a real,
if modest, position–CTR relationship) but shouldn't be treated as a
high-confidence trigger on its own — it's one signal among several, not a
standalone rule.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.